数据预处理的基础流程测试，这里提取了每个json文件下的各个子项目并合并在一起为一个.pkl & .csv，并输出到当前路径。
这个只是一个测试版本所以请暂时不要使用。
TODO：
1. 暂时使用的本地的绝对路径。后续会更改。
2. 暂时输出到的是本地的工作目录。后续会更改。
3. 暂时不确定输出的数据是否可以使用。

In [ ]:
# -*- coding: utf-8 -*-
"""
Evolutionary Heuristic Design Data Pipeline
Deliverable: Automated Data Ingestion, Cleaning & Standardization
"""

import os
import json
import pandas as pd

def fast_parse_strategy(filename):
    """Extracts prompting strategy (e.g., i1, e1) from filename conventions."""
    parts = filename.split('_')
    try:
        if 'op' in parts:
            idx = parts.index('op')
            return parts[idx + 1]
    except (ValueError, IndexError):
        pass
    return "unknown"

def scan_all_heuristic_datasets(root_path):
    """Crawls directory tree to extract algorithm, code, and objective from JSONs."""
    all_data = []
    print(f"Starting directory scan at: {root_path}")
    
    for root, dirs, files in os.walk(root_path):
        if os.path.basename(root) == 'all_programs':
            path_parts = root.replace('\\', '/').split('/')
            # Infer App Type and Scale from path depth
            app_type = path_parts[-4] if len(path_parts) >= 4 else "Unknown"
            instance_scale = path_parts[-3] if len(path_parts) >= 3 else "Unknown"
            
            print(f"Found data source: [{app_type}] -> [{instance_scale}]")
            
            for filename in os.listdir(root):
                if filename.endswith('.json') and '_Exception' not in filename:
                    file_path = os.path.join(root, filename)
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            content = json.load(f)
                            offspring = content.get('offspring', {})
                            if not offspring: continue
                            
                            algo_data = offspring.get('algorithm', [""])
                            algo = algo_data[0] if isinstance(algo_data, list) else algo_data
                            code_data = offspring.get('code', [])
                            code = "".join(code_data) if isinstance(code_data, list) else code_data
                            obj = offspring.get('objective', None)
                            
                            all_data.append({
                                'raw_app_type': app_type,
                                'instance_scale': instance_scale,
                                'filename': filename,
                                'strategy': fast_parse_strategy(filename),
                                'algorithm': algo.strip('{} '),
                                'code': code,
                                'objective': obj
                            })
                    except (json.JSONDecodeError, OSError):
                        continue
    return pd.DataFrame(all_data)

def clean_dataset(df):
    """Removes short/empty code, missing objectives, and semantic duplicates."""
    initial_count = len(df)
    
    # 1. Filter by code length (ensure logic exists)
    df = df[df['code'].str.len() > 50].copy()
    
    # 2. Drop rows with missing performance values
    df['objective'] = pd.to_numeric(df['objective'], errors='coerce')
    df = df.dropna(subset=['objective'])
    
    # 3. Deduplication: Keep only unique code per task
    # This is critical for similarity analysis to avoid bias from 'clones'
    df = df.drop_duplicates(subset=['code', 'raw_app_type'], keep='first')
    
    final_count = len(df)
    print(f"Cleaning Report: {initial_count} -> {final_count} (Removed {initial_count - final_count} invalid/duplicates)")
    return df

def finalize_task_name(row):
    """Maps inconsistent directory names to standardized labels."""
    name_map = {
        'bin_greedy': 'BinPacking',
        'cvrp_lns': 'CVRP',
        'premarshalling_astar': 'Premarshalling',
        'puzzle_astar': 'SlidingPuzzle',
        'HotAI Material': '' 
    }
    base_name = name_map.get(row['raw_app_type'], row['raw_app_type'])
    if base_name == '':
        for key, standard_name in name_map.items():
            if key != 'HotAI Material' and key in row['instance_scale']:
                return standard_name
        return row['instance_scale']
    return base_name

# --- EXECUTION ---
root_material = r"F:\KIT\HotAI\18151101\HotAI Material"

try:
    current_script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_script_dir = os.getcwd()

# 1. Ingestion
df_raw = scan_all_heuristic_datasets(root_material)

if not df_raw.empty:
    # 2. Cleaning & Deduplication
    df_final = clean_dataset(df_raw)
    
    # 3. Standardization
    df_final['task_name'] = df_final.apply(finalize_task_name, axis=1)
    
    # 4. Timeout Identification
    df_final['is_timeout'] = False
    timeout_threshold = 10000 
    df_final.loc[
        (df_final['task_name'].str.contains('Puzzle|Premarshalling', case=False)) & 
        (df_final['objective'] > timeout_threshold), 
        'is_timeout'
    ] = True
# 5. Persistence (Simplified to Current Working Directory)
    # Using relative paths ensures files are saved exactly where the script runs
    pickle_path = "all_heuristics_dataset.pkl"
    csv_path = "all_heuristics_dataset.csv"
    
    # Save the cleaned and standardized dataframe
    df_final.to_pickle(pickle_path)
    df_final.to_csv(csv_path, index=False)

    # 6. Final Report Summary
    # Getting absolute path just for the print statement to show you exactly where they are
    abs_pickle = os.path.abspath(pickle_path)
    
    print("\n" + "="*30)
    print("FINAL DATA INTEGRATION REPORT")
    print("="*30)
    print(f"Total processed samples: {len(df_final)}")
    print("\nSamples per Standardized Task:")
    print(df_final['task_name'].value_counts())
    print("\nTimeout Statistics:")
    print(df_final['is_timeout'].value_counts())
    print("="*30)
    print(f"SUCCESS: Dataset saved to current directory:\n{abs_pickle}")

Starting directory scan at: F:\KIT\HotAI\18151101\HotAI Material
Found data source: [HotAI Material] -> [bin_greedy]
Found data source: [HotAI Material] -> [bin_greedy]
Found data source: [HotAI Material] -> [bin_greedy]
Found data source: [cvrp_lns] -> [500 nodes]
Found data source: [HotAI Material] -> [premarshalling_astar]
Found data source: [HotAI Material] -> [premarshalling_astar]
Found data source: [HotAI Material] -> [premarshalling_astar]
Found data source: [HotAI Material] -> [puzzle_astar]
Found data source: [HotAI Material] -> [puzzle_astar]
Found data source: [HotAI Material] -> [puzzle_astar]
Cleaning Report: 15569 -> 14840 (Removed 729 invalid/duplicates)

FINAL DATA INTEGRATION REPORT
Total processed samples: 14840

Samples per Standardized Task:
task_name
SlidingPuzzle     4542
BinPacking        4426
Premarshalling    4315
CVRP              1557
Name: count, dtype: int64

Timeout Statistics:
is_timeout
False    14840
Name: count, dtype: int64
